# Support Vector Classifier Training with Polars

This notebook keeps the flow simple:
- load the prepared parquet files
- separate features and target
- validate the numeric training inputs
- train and evaluate the model
- save the model and CSV outputs


## 1. Imports and Config


In [1]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.svm import SVC
import joblib

TRAIN_FILE = "../data/processed/train.parquet"
TEST_FILE = "../data/processed/test.parquet"
MODEL_DIR = "../model"
RESULTS_DIR = "../model/results"
TARGET_COLUMN = "admitted"
RANDOM_SEED = 42
MODEL_NAME = "svc"

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

MODEL_FILE = Path(MODEL_DIR) / "svc_model.joblib"
RESULTS_FILE = Path(RESULTS_DIR) / "svc_results.csv"
METRICS_FILE = Path(RESULTS_DIR) / "svc_metrics.csv"


## 2. Load the Dataset


In [2]:
for required_file in [TRAIN_FILE, TEST_FILE]:
    if not Path(required_file).exists():
        raise FileNotFoundError(f"Expected parquet file at: {required_file}")

train_df = pl.read_parquet(TRAIN_FILE)
test_df = pl.read_parquet(TEST_FILE)

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
train_df.head()


train shape: (56, 8)
test shape: (16, 8)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days,admitted
i64,i64,i64,f64,i64,i64,i64,i64
34,2,4,0.47,2,6,21,1
35,3,3,0.65,1,2,19,1
71,7,0,0.41,1,5,3,1
72,7,1,0.53,5,3,15,1
49,8,2,0.65,2,9,17,1


## 3. Separate Features and Target

This notebook uses the prepared train and test parquet files exactly as they come from the upstream split notebook.


In [3]:
if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise ValueError(f"Both parquet files must include the target column: {TARGET_COLUMN}")

feature_columns = [column_name for column_name in train_df.columns if column_name != TARGET_COLUMN]

X_train = train_df.select(feature_columns)
X_test = test_df.select([column_name for column_name in test_df.columns if column_name != TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN].cast(pl.Int32)
y_test = test_df[TARGET_COLUMN].cast(pl.Int32)

print(feature_columns)
print(X_train.shape)
X_train.head()


['age', 'length_of_stay', 'prior_admissions', 'lab_score', 'comorbidity_count', 'medications_count', 'followup_gap_days']
(56, 7)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days
i64,i64,i64,f64,i64,i64,i64
34,2,4,0.47,2,6,21
35,3,3,0.65,1,2,19
71,7,0,0.41,1,5,3
72,7,1,0.53,5,3,15
49,8,2,0.65,2,9,17


## 4. Validate the Prepared Features

These notebooks expect the upstream split and feature-selection notebook to output numeric, model-ready feature columns.


In [4]:
test_feature_columns = [column_name for column_name in test_df.columns if column_name != TARGET_COLUMN]

if feature_columns != test_feature_columns:
    raise ValueError(
        "Train and test parquet files must have the same feature columns in the same order."
    )

non_numeric_columns = sorted(
    set(
        [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
        + [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
    )
)

if non_numeric_columns:
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric columns found: {non_numeric_columns}"
    )

print(f"validated {len(feature_columns)} numeric feature columns")


validated 7 numeric feature columns


## 5. Train the Model


In [5]:
model = SVC(kernel="rbf", probability=True, random_state=RANDOM_SEED)
model.fit(X_train.to_numpy(), y_train.to_numpy())

model


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


## 6. Evaluate and Save


In [6]:
probabilities = model.predict_proba(X_test.to_numpy())[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test.to_numpy()

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = test_df.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


model saved to: ..\model\svc_model.joblib
metrics saved to: ..\model\results\svc_metrics.csv
results saved to: ..\model\results\svc_results.csv


model,accuracy,precision,recall,roc_auc
str,f64,f64,f64,f64
"""svc""",0.9375,0.9375,1.0,1.0


## 7. Preview Results


In [7]:
results_frame.head(10)


age,length_of_stay,prior_admissions,lab_score,comorbidity_count,medications_count,followup_gap_days,admitted,actual,predicted,probability,model
i64,i64,i64,f64,i64,i64,i64,i64,i32,i64,f64,str
48,7,3,0.47,3,4,19,1,1,1,0.958936,"""svc"""
30,1,2,0.59,3,7,7,1,1,1,0.937713,"""svc"""
53,8,1,0.47,6,10,5,1,1,1,0.970802,"""svc"""
35,2,0,0.59,6,4,13,1,1,1,0.948555,"""svc"""
43,6,0,0.47,6,7,13,1,1,1,0.962557,"""svc"""
31,2,1,0.77,2,3,5,0,0,1,0.922767,"""svc"""
34,3,2,0.53,3,4,7,1,1,1,0.940228,"""svc"""
49,7,4,0.59,1,2,11,1,1,1,0.954679,"""svc"""
61,4,1,0.35,6,4,5,1,1,1,0.962498,"""svc"""
